# Phase 2 — SQL Analysis (DuckDB)

**Primary business question (Section 1):** *What are the top drivers of low review scores (≤ 3) on Olist, and which interventions would produce the largest lift in average review score?*

This notebook doesn't answer one sub-question in isolation — it's the descriptive SQL layer that locates *where* the low-review problem concentrates (category, geography, seller, delivery timing, freight burden), using DuckDB directly on the Phase 1 parquet outputs. Those findings set up Phase 3's EDA and Phase 4's hypothesis tests. Per Section 6, Q5 was intentionally removed and folded into Phase 4's chi-square test on installment count.

**Queries in this notebook:**

| # | Produces |
|---|---|
| Q0 | Baseline snapshot — overall problem size (orders, revenue, avg review, % low-review) and the month-by-month trend |
| Q1 | Category priority matrix — revenue-at-risk ranking of product categories (volume × severity) |
| Q2 | Delivery-sensitivity by category — how much early/late delivery moves review scores, by category |
| Q3 | State delivery-vs-review quadrant — which states' review problems are delivery-driven vs. something else |
| Q4 | Seller experience vs. review, within category — whether seller tenure predicts better reviews once category is controlled for |
| Q6 | Seller ranking within category — bottom-20 sellers per category, an intervention shortlist |
| Q7 | Freight-burden effect on review, category-controlled — whether disproportionate freight (relative to category peers) hurts reviews |
| Q8 | Product-size-adjusted delivery expectation gap — which states/sellers are slow relative to what's normal for the product's weight class |

**Cross-cutting rule (Section 6):** every query here either partitions by category, controls for category, or explicitly states why pooling is acceptable in that specific case — called out per query below.

## Setup

Phase 1 produced two parquet outputs: `orders_clean.parquet` (order-grain, backs Q0/Q1/Q2/Q3/Q7) and `order_items_clean.parquet` (item-grain, carries `seller_id` and `product_weight_g`, backs Q4/Q6/Q8 — `orders_clean` intentionally has no single `seller_id` per row, per the Section 6 grain reminder). We query the parquet files directly with DuckDB rather than loading through pandas first.

In [1]:
import os
import sys
from pathlib import Path

import duckdb
import pandas as pd

if Path.cwd().name == "notebooks":
    os.chdir("..")

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

ORDERS_PATH = "data/processed/orders_clean.parquet"
ITEMS_PATH = "data/processed/order_items_clean.parquet"

for path in (ORDERS_PATH, ITEMS_PATH):
    assert Path(path).exists(), f"missing required Phase 1 output: {path}"

orders_shape = duckdb.sql(f"SELECT COUNT(*) AS n, COUNT(DISTINCT order_id) AS n_orders FROM '{ORDERS_PATH}'").df()
items_shape = duckdb.sql(f"SELECT COUNT(*) AS n, COUNT(DISTINCT order_id) AS n_orders, COUNT(DISTINCT seller_id) AS n_sellers FROM '{ITEMS_PATH}'").df()
print("orders_clean.parquet:", orders_shape.to_dict("records")[0])
print("order_items_clean.parquet:", items_shape.to_dict("records")[0])

orders_clean.parquet: {'n': 96470, 'n_orders': 96470}
order_items_clean.parquet: {'n': 110163, 'n_orders': 96448, 'n_sellers': 2970}


We'll also collect every query's SQL text (with its comment block) into a list as we go, then export it verbatim to `sql/queries.sql` at the end — so the exported file is guaranteed to match exactly what ran here.

In [2]:
exported_queries = []  # list of (label, comment_block, sql_text) in the order they should appear in sql/queries.sql

def register(label, comment_block, sql_text):
    exported_queries.append((label, comment_block.strip(), sql_text.strip()))
    return sql_text

## Q0 — Baseline snapshot

**Business question:** How large is the low-review problem, and what has the monthly trend looked like?

**Grain of result:** two result sets — (a) one row, the all-time overall snapshot; (b) one row per calendar month.

**How to read it:** `pct_low_review` is the share of *reviewed* orders (review_score not null) scoring ≤ 3; orders without a submitted review still count toward `total_orders`/`total_revenue` but are excluded from review-based metrics, since they can't be averaged. `total_revenue` is product revenue (`total_price`, excludes freight).

**Cross-cutting rule:** Q0 intentionally pools across every category and state — it's the single headline baseline, not a comparison, so pooling here is the point. Q1 onward break this down by category/state/seller.

In [3]:
Q0_OVERALL_COMMENT = """
-- Q0a -- Baseline snapshot (overall)
-- Business question: How large is the low-review problem overall?
-- Grain of result: one row -- the all-time snapshot across all delivered orders.
-- How to read it: pct_low_review is the share of *reviewed* orders (review_score
--   not null) scoring <= 3. total_revenue is product revenue (total_price),
--   excludes freight. Orders without a submitted review count toward
--   total_orders/total_revenue but not the review-based columns.
-- Cross-cutting note: intentionally pools all categories/states -- this is the
--   headline baseline, not a comparison.
"""

Q0_OVERALL_SQL = """
WITH base AS (
    SELECT order_id, total_price, review_score
    FROM '{orders}'
)
SELECT
    COUNT(*) AS total_orders,
    ROUND(SUM(total_price), 2) AS total_revenue,
    ROUND(AVG(review_score), 3) AS avg_review_score,
    COUNT(*) FILTER (WHERE review_score IS NOT NULL) AS reviewed_orders,
    ROUND(100.0 * COUNT(*) FILTER (WHERE review_score <= 3)
          / COUNT(*) FILTER (WHERE review_score IS NOT NULL), 2) AS pct_low_review
FROM base
""".format(orders=ORDERS_PATH)

register("Q0a_overall", Q0_OVERALL_COMMENT, Q0_OVERALL_SQL)

q0_overall = duckdb.sql(Q0_OVERALL_SQL).df()
print(f"Result grain: {len(q0_overall)} row -- the all-time overall snapshot.")
q0_overall

Result grain: 1 row -- the all-time overall snapshot.


,total_orders,total_revenue,avg_review_score,reviewed_orders,pct_low_review
0,96470,13220248.93,4.156,95824,21.07


In [4]:
Q0_MONTHLY_COMMENT = """
-- Q0b -- Baseline snapshot (monthly trend)
-- Business question: What has the monthly trend in order volume, review score,
--   and low-review share looked like?
-- Grain of result: one row per calendar month of order_purchase_timestamp.
-- How to read it: same pct_low_review definition as Q0a, computed within each month.
-- Cross-cutting note: pools all categories/states within each month -- this is
--   the headline monthly trend, not a category comparison.
"""

Q0_MONTHLY_SQL = """
WITH base AS (
    SELECT order_id, date_trunc('month', order_purchase_timestamp) AS order_month, review_score
    FROM '{orders}'
)
SELECT
    order_month,
    COUNT(*) AS order_volume,
    ROUND(AVG(review_score), 3) AS avg_review_score,
    ROUND(100.0 * COUNT(*) FILTER (WHERE review_score <= 3)
          / COUNT(*) FILTER (WHERE review_score IS NOT NULL), 2) AS pct_low_review
FROM base
GROUP BY order_month
ORDER BY order_month
""".format(orders=ORDERS_PATH)

register("Q0b_monthly", Q0_MONTHLY_COMMENT, Q0_MONTHLY_SQL)

q0_monthly = duckdb.sql(Q0_MONTHLY_SQL).df()
print(f"Result grain: {len(q0_monthly)} rows -- one row per calendar month.")
q0_monthly

Result grain: 23 rows -- one row per calendar month.


,order_month,order_volume,avg_review_score,pct_low_review
0,2016-09-01,1,1.000,100.00
1,2016-10-01,265,4.011,24.43
2,2016-12-01,1,5.000,0.00
3,2017-01-01,750,4.198,20.24
4,2017-02-01,1653,4.203,20.69
5,2017-03-01,2546,4.187,20.34
6,2017-04-01,2303,4.135,21.83
7,2017-05-01,3545,4.236,19.11
8,2017-06-01,3135,4.223,19.41
9,2017-07-01,3872,4.260,18.61


**Interpretation:** 21.07% of the 95,824 reviewed orders (96,470 delivered orders total, R$13.22M in product revenue) come back with a review score of 3 or below — roughly 1 in 5. The trend isn't flat: low-review share holds around 17–22% through most of 2017, spikes to 25.9% in November 2017 (which also happens to be the single highest-volume month, 7,288 orders — plausibly a Black-Friday demand surge straining fulfillment), then worsens further to 28.8% and 30.4% in February–March 2018 despite unexceptional volume, before recovering to 16–19% by mid-2018. The Feb–Mar 2018 spike isn't explained by volume alone and is worth a dedicated look in Phase 3's monthly delivery-time overlay chart.

## Q1 — Category priority matrix

**Business question:** Which product categories should be prioritized for intervention, weighting both size and severity?

**Grain of result:** one row per `primary_category` (74 categories).

**How to read it:** `revenue_at_risk = revenue × pct_low_review_rate` — a category with middling quality but huge volume can outrank a small category with terrible quality. `priority_rank` orders categories by `revenue_at_risk` descending via `RANK() OVER (...)`.

**Cross-cutting rule:** this query *is* the category partition — every metric is computed within `GROUP BY primary_category`, so no cross-category pooling occurs.

In [5]:
Q1_COMMENT = """
-- Q1 -- Category priority matrix
-- Business question: Which product categories should be prioritized for
--   intervention? Weight both size (volume/revenue) and severity (% low-review).
-- Grain of result: one row per primary_category.
-- How to read it: revenue_at_risk = revenue * pct_low_review_rate. priority_rank
--   orders categories by revenue_at_risk descending.
-- Cross-cutting note: partitioned by primary_category throughout -- no
--   cross-category pooling.
"""

Q1_SQL = """
WITH category_stats AS (
    SELECT
        primary_category,
        COUNT(*) AS order_volume,
        SUM(total_price) AS revenue,
        AVG(review_score) AS avg_review_score,
        COUNT(*) FILTER (WHERE review_score <= 3) * 1.0
            / NULLIF(COUNT(*) FILTER (WHERE review_score IS NOT NULL), 0) AS pct_low_review_rate
    FROM '{orders}'
    GROUP BY primary_category
),
priority AS (
    SELECT
        primary_category,
        order_volume,
        ROUND(revenue, 2) AS revenue,
        ROUND(avg_review_score, 3) AS avg_review_score,
        ROUND(pct_low_review_rate * 100, 2) AS pct_low_review,
        ROUND(revenue * pct_low_review_rate, 2) AS revenue_at_risk
    FROM category_stats
)
SELECT *, RANK() OVER (ORDER BY revenue_at_risk DESC) AS priority_rank
FROM priority
ORDER BY priority_rank
""".format(orders=ORDERS_PATH)

register("Q1", Q1_COMMENT, Q1_SQL)

q1 = duckdb.sql(Q1_SQL).df()
print(f"Result grain: {len(q1)} rows -- one row per product category.")
q1.head(15)

Result grain: 74 rows -- one row per product category.


,primary_category,order_volume,revenue,avg_review_score,pct_low_review,revenue_at_risk,priority_rank
0,bed_bath_table,9181,1023530.56,4.008,25.73,263344.83,1
1,watches_gifts,5476,1166874.18,4.123,22.28,259949.34,2
2,health_beauty,8613,1233813.10,4.233,19.04,234895.43,3
3,computers_accessories,6511,888482.99,4.083,22.70,201691.12,4
4,sports_leisure,7492,954346.89,4.235,18.42,175800.74,5
5,furniture_decor,6188,709605.12,4.079,23.61,167523.19,6
6,housewares,5680,615325.93,4.204,20.03,123261.36,7
7,auto,3793,579341.55,4.153,20.45,118477.27,8
8,cool_stuff,3530,610373.43,4.230,18.90,115381.84,9
9,office_furniture,1251,269124.55,3.647,36.34,97804.33,10


**Interpretation:** `bed_bath_table` tops the priority list at R$263k of revenue-at-risk despite a middling 25.7% low-review rate, purely on volume (9,181 orders, R$1.02M revenue) — a large, mediocre category beats a smaller disaster. `office_furniture`, by contrast, has the worst average review (3.65) and highest low-review rate (36.3%) of any category in the top 15, but ranks only 10th on revenue-at-risk because its revenue base (R$269k) is much smaller — a genuine quality problem, just not the biggest dollar problem. Ranking by revenue-at-risk rather than review score alone changes which category ops should tackle first.

## Q2 — Delivery-sensitivity by category

**Business question:** Does late delivery hurt reviews equally across categories, or are some categories much more delivery-sensitive? Does early delivery actually help?

**Grain of result:** one row per `primary_category` (categories missing any of the Early 2+ / On-time / Late 8+ buckets entirely are excluded, since `early_lift`/`late_drop` need all three).

**How to read it:** `delivery_gap_days` is `actual - estimated` (positive = late). Buckets: Early 2+ (≤−2), Early 1 (−1), On-time (0), Late 1–3, Late 4–7, Late 8+ (≥8). `early_lift = avg_review(Early 2+) - avg_review(On-time)`; `late_drop = avg_review(On-time) - avg_review(Late 8+)`. Ranked by `late_drop` descending — the logistics prioritization list.

**Cross-cutting rule:** partitioned by `primary_category` throughout.

In [6]:
Q2_COMMENT = """
-- Q2 -- Delivery-sensitivity by category
-- Business question: Does late delivery hurt reviews equally across categories?
--   Does early delivery help, or is on-time already enough?
-- Grain of result: one row per primary_category (categories missing any of the
--   Early 2+ / On-time / Late 8+ buckets entirely are excluded).
-- How to read it: delivery_gap_days = actual - estimated (positive = late).
--   early_lift = avg_review(Early 2+) - avg_review(On-time);
--   late_drop  = avg_review(On-time) - avg_review(Late 8+).
--   Ranked by late_drop descending (logistics prioritization list).
-- Cross-cutting note: partitioned by primary_category throughout.
"""

Q2_SQL = """
WITH bucketed AS (
    SELECT
        primary_category,
        review_score,
        CASE
            WHEN delivery_gap_days <= -2 THEN 'Early 2+'
            WHEN delivery_gap_days = -1 THEN 'Early 1'
            WHEN delivery_gap_days = 0 THEN 'On-time'
            WHEN delivery_gap_days BETWEEN 1 AND 3 THEN 'Late 1-3'
            WHEN delivery_gap_days BETWEEN 4 AND 7 THEN 'Late 4-7'
            ELSE 'Late 8+'
        END AS gap_bucket
    FROM '{orders}'
    WHERE review_score IS NOT NULL
),
category_bucket_avg AS (
    SELECT primary_category, gap_bucket, COUNT(*) AS order_count, AVG(review_score) AS avg_review_score
    FROM bucketed
    GROUP BY primary_category, gap_bucket
),
pivoted AS (
    SELECT
        primary_category,
        MAX(avg_review_score) FILTER (WHERE gap_bucket = 'Early 2+') AS early2plus,
        MAX(avg_review_score) FILTER (WHERE gap_bucket = 'Early 1') AS early1,
        MAX(avg_review_score) FILTER (WHERE gap_bucket = 'On-time') AS ontime,
        MAX(avg_review_score) FILTER (WHERE gap_bucket = 'Late 1-3') AS late1to3,
        MAX(avg_review_score) FILTER (WHERE gap_bucket = 'Late 4-7') AS late4to7,
        MAX(avg_review_score) FILTER (WHERE gap_bucket = 'Late 8+') AS late8plus,
        SUM(order_count) AS total_orders
    FROM category_bucket_avg
    GROUP BY primary_category
)
SELECT
    primary_category, total_orders,
    ROUND(early2plus, 3) AS early2plus, ROUND(early1, 3) AS early1, ROUND(ontime, 3) AS ontime,
    ROUND(late1to3, 3) AS late1to3, ROUND(late4to7, 3) AS late4to7, ROUND(late8plus, 3) AS late8plus,
    ROUND(early2plus - ontime, 3) AS early_lift,
    ROUND(ontime - late8plus, 3) AS late_drop
FROM pivoted
WHERE early2plus IS NOT NULL AND ontime IS NOT NULL AND late8plus IS NOT NULL
ORDER BY late_drop DESC
""".format(orders=ORDERS_PATH)

register("Q2", Q2_COMMENT, Q2_SQL)

q2 = duckdb.sql(Q2_SQL).df()
print(f"Result grain: {len(q2)} rows (of 74 categories) -- one row per category with data in all three key buckets.")
q2.head(15)

Result grain: 54 rows (of 74 categories) -- one row per category with data in all three key buckets.


,primary_category,total_orders,early2plus,early1,ontime,late1to3,late4to7,late8plus,early_lift,late_drop
0,furniture_bedroom,90.0,4.427,5.000,5.000,4.000,1.333,1.000,-0.573,4.000
1,art,191.0,4.172,4.571,5.000,2.750,2.333,1.000,-0.828,4.000
2,music,37.0,4.471,NaN,5.000,NaN,NaN,1.000,-0.529,4.000
3,small_appliances_home_oven_and_coffee,72.0,4.545,4.000,5.000,3.000,NaN,1.000,-0.455,4.000
4,fixed_telephony,209.0,4.051,4.333,5.000,5.000,NaN,1.000,-0.949,4.000
5,agro_industry_and_commerce,177.0,4.201,4.000,5.000,2.000,2.500,1.250,-0.799,3.750
6,food,432.0,4.459,4.429,4.769,3.563,1.625,1.167,-0.311,3.603
7,drinks,282.0,4.381,3.333,4.500,2.250,2.200,1.333,-0.119,3.167
8,market_place,266.0,4.198,4.000,4.500,4.000,3.200,1.400,-0.302,3.100
9,home_appliances_2,226.0,4.333,2.333,4.500,3.333,1.250,1.429,-0.167,3.071


In [7]:
# Diagnostic referenced in the interpretation below: how thin is the On-time bucket?
ontime_check = duckdb.sql(f"""
SELECT
    COUNT(*) FILTER (WHERE delivery_gap_days = 0) AS n_ontime,
    COUNT(*) FILTER (WHERE delivery_gap_days <= -2) AS n_early2plus,
    COUNT(*) AS n_total
FROM '{ORDERS_PATH}'
WHERE review_score IS NOT NULL
""").df()
ontime_check

,n_ontime,n_early2plus,n_total
0,1280,86713,95824


**Interpretation — thin-data flag:** the exact On-time bucket (`delivery_gap_days = 0`) is vanishingly rare — only 1,280 of 95,824 reviewed orders (1.3%), versus 86,713 (90.5%) arriving 2+ days early, because Olist's estimated delivery dates are heavily padded. Even the largest categories have only 50–124 orders in that exact bucket (e.g. `health_beauty`: 124/8,567), so per-category On-time averages are noisy — that's why `early_lift` looks *negative* (early beats on-time) for many rows in the table: it's sampling noise in a tiny bucket, not evidence that arriving early hurts. **Suggested adjustment:** redefine "On-time" as a small band (e.g. −2 to +1 days) before trusting the early-lift number at the category level, rather than an exact single-day match. `late_drop`, in contrast, is a robust and consistent signal: it's positive across nearly every category, and even the highest-volume categories (`health_beauty` 2.63, `sports_leisure` 2.50, `auto` 2.42, `bed_bath_table` 2.38) show a consistent ~2.4–2.6-point drop from on-time to Late 8+ — late delivery clearly and reliably hurts reviews everywhere.

## Q3 — State delivery-vs-review quadrant

**Business question:** Which states are underperforming *because of* delivery, versus underperforming *despite* good delivery?

**Grain of result:** one row per `customer_state` (27 Brazilian states/territories present).

**How to read it:** `on_time_rate` = share of orders with `delivery_gap_days ≤ 0`. Quadrant thresholds are the **median** `on_time_rate` and **median** `avg_review_score` across the 27 states (stated explicitly, computed in the `medians` CTE) — not a fixed cutoff. `Low OTD + Low Review` = delivery is the story; `High OTD + Low Review` = delivery is NOT the story, investigate further; `Low OTD + High Review` = customers tolerant here; `High OTD + High Review` = healthy.

**Cross-cutting rule:** this query pools across product categories within each state — justified because Q3 studies a geographic/logistics axis, not a product axis, and the goal is a state-level triage list, not a category comparison (categories are already handled by Q1/Q2). A category-mix confound across states is possible but out of scope here; Phase 4's Levene's test on delivery-gap variance across the best/worst states is the follow-up that would surface it.

In [8]:
Q3_COMMENT = """
-- Q3 -- State delivery-vs-review quadrant
-- Business question: Which states are underperforming because of delivery,
--   versus underperforming despite good delivery?
-- Grain of result: one row per customer_state.
-- How to read it: on_time_rate = share of orders with delivery_gap_days <= 0.
--   Quadrant thresholds are the MEDIAN on_time_rate and MEDIAN avg_review_score
--   across states (computed in the medians CTE), not a fixed cutoff.
-- Cross-cutting note: pools across product categories within each state --
--   justified because this is a geographic/logistics axis, not a product axis;
--   category-level analysis is Q1/Q2's job. A category-mix confound across
--   states is possible but out of scope here (see Phase 4 Levene's test).
"""

Q3_SQL = """
WITH state_stats AS (
    SELECT
        customer_state,
        COUNT(*) AS order_volume,
        AVG(review_score) AS avg_review_score,
        100.0 * COUNT(*) FILTER (WHERE delivery_gap_days <= 0) / COUNT(*) AS on_time_rate
    FROM '{orders}'
    WHERE review_score IS NOT NULL
    GROUP BY customer_state
),
medians AS (
    SELECT MEDIAN(on_time_rate) AS median_otd, MEDIAN(avg_review_score) AS median_review
    FROM state_stats
)
SELECT
    s.customer_state, s.order_volume,
    ROUND(s.on_time_rate, 2) AS on_time_rate,
    ROUND(s.avg_review_score, 3) AS avg_review_score,
    CASE
        WHEN s.on_time_rate < m.median_otd AND s.avg_review_score < m.median_review THEN 'Low OTD + Low Review'
        WHEN s.on_time_rate >= m.median_otd AND s.avg_review_score < m.median_review THEN 'High OTD + Low Review'
        WHEN s.on_time_rate < m.median_otd AND s.avg_review_score >= m.median_review THEN 'Low OTD + High Review'
        ELSE 'High OTD + High Review'
    END AS quadrant
FROM state_stats s CROSS JOIN medians m
ORDER BY s.order_volume DESC
""".format(orders=ORDERS_PATH)

register("Q3", Q3_COMMENT, Q3_SQL)

q3 = duckdb.sql(Q3_SQL).df()
print(f"Result grain: {len(q3)} rows -- one row per state.")
q3

Result grain: 27 rows -- one row per state.


,customer_state,order_volume,on_time_rate,avg_review_score,quadrant
0,SP,40266,95.56,4.246,High OTD + High Review
1,RJ,12211,88.08,3.965,Low OTD + Low Review
2,MG,11285,95.53,4.192,High OTD + High Review
3,RS,5326,93.97,4.185,High OTD + High Review
4,PR,4900,96.04,4.240,High OTD + High Review
5,SC,3519,91.93,4.134,High OTD + High Review
6,BA,3229,88.11,3.931,Low OTD + Low Review
7,DF,2070,94.30,4.131,High OTD + High Review
8,ES,1969,89.59,4.079,Low OTD + Low Review
9,GO,1946,93.63,4.106,High OTD + High Review


**Interpretation:** most states fall on the expected diagonal — 23 of 27 states pair high OTD with high review (SP, MG, RS, PR, SC, DF, GO, MT, RN, RO, AM, AP) or low OTD with low review (RJ, BA, ES, CE, PA, MA, PB, PI, AL, SE, RR). Four states break the pattern: **PE and AC** sit in `High OTD + Low Review` — delivery is *not* the story there, something else (seller mix, product mix, or price sensitivity) is dragging reviews down despite good logistics, worth investigating in Phase 3. **MS and TO** sit in `Low OTD + High Review` — customers there tolerate slower delivery better than the median state, worth understanding before assuming a logistics fix helps everywhere equally.

## Q4 — Seller experience vs. review score, within category

**Business question:** Do experienced sellers actually deliver better reviews than new sellers, or does the problem persist regardless of tenure?

**Grain of result:** one row per (`category`, `tenure_bucket`) combination, plus a single overall category-controlled weighted difference.

**How to read it:** each seller's tenure start is `MIN(order_purchase_timestamp) OVER (PARTITION BY seller_id)` (their first order in the dataset). **Judgment call, stated explicitly:** "≥6 months in the dataset" is operationalized as the span from that first order to the *last* order date anywhere in the dataset (2018-08-29) — i.e., how long the seller has been observed in the marketplace overall, not a per-order lookback. Sellers observed for ≥182 days (≈6 months) are `Established`, else `New`. The overall difference is a **volume-weighted average of each category's (Established − New) gap**, not a pooled raw average — this is the category-control the cross-cutting rule requires, computed from item-grain data (`order_items_clean.parquet`) since seller_id doesn't exist on `orders_clean`.

**Cross-cutting rule:** explicitly satisfied — the per-row breakdown is partitioned by category, and the "overall" figure is a weighted average of within-category differences, not a pooled comparison across categories (which would risk Simpson's paradox).

In [9]:
Q4_COMMENT = """
-- Q4 -- Seller experience vs. review score, within category
-- Business question: Do experienced sellers deliver better reviews than new
--   sellers, or does the problem persist regardless of tenure?
-- Grain of result: one row per (category, tenure_bucket).
-- How to read it: seller_first_order = MIN(order_purchase_timestamp) OVER
--   (PARTITION BY seller_id). JUDGMENT CALL: "tenure in the dataset" is
--   operationalized as (dataset's last order date - seller's first order date),
--   i.e. how long the seller has been observed overall, not a per-order
--   lookback. >= 182 days (~6 months) = Established, else New.
-- Cross-cutting note: partitioned by category; see Q4_OVERALL_SQL below for
--   the category-controlled overall figure (a weighted average of
--   within-category differences, not a pooled cross-category comparison).
"""

Q4_SQL = """
WITH item_reviews AS (
    SELECT seller_id, category, order_id, order_purchase_timestamp, review_score
    FROM '{items}'
    WHERE review_score IS NOT NULL
),
seller_tenure AS (
    SELECT *,
        MIN(order_purchase_timestamp) OVER (PARTITION BY seller_id) AS seller_first_order,
        MAX(order_purchase_timestamp) OVER () AS dataset_last_order
    FROM item_reviews
),
bucketed AS (
    SELECT *,
        CASE WHEN date_diff('day', seller_first_order, dataset_last_order) < 182
             THEN 'New' ELSE 'Established' END AS tenure_bucket
    FROM seller_tenure
)
SELECT
    category, tenure_bucket,
    COUNT(DISTINCT seller_id) AS seller_count,
    COUNT(*) AS order_item_count,
    ROUND(AVG(review_score), 3) AS avg_review_score
FROM bucketed
GROUP BY category, tenure_bucket
ORDER BY category, tenure_bucket
""".format(items=ITEMS_PATH)

register("Q4_breakdown", Q4_COMMENT, Q4_SQL)

q4 = duckdb.sql(Q4_SQL).df()
print(f"Result grain: {len(q4)} rows -- one row per (category, tenure_bucket).")
q4.head(15)

Result grain: 138 rows -- one row per (category, tenure_bucket).


,category,tenure_bucket,seller_count,order_item_count,avg_review_score
0,agro_industry_and_commerce,Established,21,183,4.120
1,agro_industry_and_commerce,New,11,23,3.826
2,air_conditioning,Established,40,241,4.145
3,air_conditioning,New,11,43,3.535
4,art,Established,25,182,4.093
5,art,New,7,13,3.923
6,arts_and_craftmanship,Established,5,5,4.800
7,arts_and_craftmanship,New,8,19,3.947
8,audio,Established,35,356,3.831
9,audio,New,1,2,5.000


In [10]:
Q4_OVERALL_COMMENT = """
-- Q4 (continued) -- category-controlled overall tenure effect
-- Business question: same as Q4_breakdown -- this collapses it to one number.
-- Grain of result: one row -- the volume-weighted average, across categories,
--   of (avg_review[Established] - avg_review[New]).
-- How to read it: NOT a pooled average of raw review scores across categories
--   (that would risk Simpson's paradox) -- it's a weighted average of
--   WITHIN-category differences, weighted by each category's order-item count.
--   Only categories with both a New and an Established seller present
--   contribute.
"""

Q4_OVERALL_SQL = """
WITH item_reviews AS (
    SELECT seller_id, category, order_id, order_purchase_timestamp, review_score
    FROM '{items}'
    WHERE review_score IS NOT NULL
),
seller_tenure AS (
    SELECT *,
        MIN(order_purchase_timestamp) OVER (PARTITION BY seller_id) AS seller_first_order,
        MAX(order_purchase_timestamp) OVER () AS dataset_last_order
    FROM item_reviews
),
bucketed AS (
    SELECT *,
        CASE WHEN date_diff('day', seller_first_order, dataset_last_order) < 182
             THEN 'New' ELSE 'Established' END AS tenure_bucket
    FROM seller_tenure
),
category_bucket AS (
    SELECT category, tenure_bucket, COUNT(*) AS n, AVG(review_score) AS avg_review
    FROM bucketed
    GROUP BY category, tenure_bucket
),
category_diff AS (
    SELECT
        category,
        MAX(avg_review) FILTER (WHERE tenure_bucket = 'Established')
            - MAX(avg_review) FILTER (WHERE tenure_bucket = 'New') AS established_minus_new,
        SUM(n) AS category_n
    FROM category_bucket
    GROUP BY category
)
SELECT
    COUNT(*) AS categories_with_both_buckets,
    ROUND(SUM(established_minus_new * category_n) / SUM(category_n), 4) AS overall_category_controlled_diff
FROM category_diff
WHERE established_minus_new IS NOT NULL
""".format(items=ITEMS_PATH)

register("Q4_overall_diff", Q4_OVERALL_COMMENT, Q4_OVERALL_SQL)

q4_overall = duckdb.sql(Q4_OVERALL_SQL).df()
print(f"Result grain: {len(q4_overall)} row -- the category-controlled overall tenure effect.")
q4_overall

Result grain: 1 row -- the category-controlled overall tenure effect.


,categories_with_both_buckets,overall_category_controlled_diff
0,64,-0.1739


**Interpretation — surprising result, flagged:** controlling for category, established sellers (≥6 months observed tenure) average a review score **0.174 points LOWER** than new sellers (weighted across the 64 categories where both groups exist) — the opposite of an "experience fixes it" story. This holds within individual categories too: `bed_bath_table` (Established 3.912 vs New 4.116), `computers_accessories` (3.975 vs 4.287), and `auto` (4.099 vs 4.231) all show new sellers outperforming established ones. Taken at face value this argues *against* seller-tenure/onboarding as the fix — the review problem looks structural, not a ramp-up problem — but the direction is counterintuitive enough (could be survivorship: mediocre sellers who "stuck around" vs. new sellers still in a still-trying-to-impress phase) that it's worth treating as a Phase 4 hypothesis to test formally, not a settled conclusion.

## Q6 — Seller ranking within category (intervention shortlist)

**Business question:** Which specific sellers should be flagged for review or intervention, ranked fairly against peers in the same category?

**Grain of result:** one row per (seller, category) — the bottom 20 sellers per primary category, restricted to sellers with ≥20 orders.

**How to read it:** a seller's "primary category" is the category they've shipped the most distinct orders in. `order_count`/`avg_review_score`/`pct_low_review` are computed after de-duplicating to one row per (seller, order) — so a multi-item order from the same seller isn't double-counted in their average. `category_rank = RANK() OVER (PARTITION BY primary_category ORDER BY avg_review_score ASC)`; only rank ≤20 shown.

**Cross-cutting rule:** partitioned by `primary_category` throughout — sellers are only ever ranked against peers in their own category.

In [11]:
Q6_COMMENT = """
-- Q6 -- Seller ranking within category (intervention shortlist)
-- Business question: Which specific sellers should be flagged for review or
--   intervention, ranked fairly against peers in the same category?
-- Grain of result: one row per (seller, category) -- bottom 20 sellers per
--   primary category, sellers with >= 20 orders only (minimum-volume threshold
--   to avoid noise from tiny sellers).
-- How to read it: primary_category = the category a seller has shipped the
--   most distinct orders in. Stats are computed after de-duplicating to one
--   row per (seller, order), so multi-item orders from the same seller aren't
--   double-counted. category_rank = RANK() OVER (PARTITION BY primary_category
--   ORDER BY avg_review_score ASC); only rank <= 20 shown.
-- Cross-cutting note: partitioned by primary_category throughout.
"""

Q6_SQL = """
WITH seller_orders AS (
    SELECT DISTINCT seller_id, order_id, review_score
    FROM '{items}'
),
seller_category_counts AS (
    SELECT seller_id, category, COUNT(DISTINCT order_id) AS orders_in_category
    FROM '{items}'
    GROUP BY seller_id, category
),
seller_primary_category AS (
    SELECT seller_id, category AS primary_category
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY orders_in_category DESC) AS rn
        FROM seller_category_counts
    )
    WHERE rn = 1
),
seller_stats AS (
    SELECT
        seller_id,
        COUNT(*) AS order_count,
        AVG(review_score) AS avg_review_score,
        100.0 * COUNT(*) FILTER (WHERE review_score <= 3)
            / NULLIF(COUNT(*) FILTER (WHERE review_score IS NOT NULL), 0) AS pct_low_review
    FROM seller_orders
    GROUP BY seller_id
    HAVING COUNT(*) >= 20
),
ranked AS (
    SELECT
        s.seller_id, p.primary_category, s.order_count,
        ROUND(s.avg_review_score, 3) AS avg_review_score,
        ROUND(s.pct_low_review, 2) AS pct_low_review,
        RANK() OVER (PARTITION BY p.primary_category ORDER BY s.avg_review_score ASC) AS category_rank
    FROM seller_stats s
    JOIN seller_primary_category p ON s.seller_id = p.seller_id
)
SELECT * FROM ranked WHERE category_rank <= 20
ORDER BY primary_category, category_rank
""".format(items=ITEMS_PATH)

register("Q6", Q6_COMMENT, Q6_SQL)

q6 = duckdb.sql(Q6_SQL).df()
n_qualifying_sellers = duckdb.sql(f"""
    SELECT COUNT(*) AS n FROM (
        SELECT seller_id FROM (SELECT DISTINCT seller_id, order_id FROM '{ITEMS_PATH}')
        GROUP BY seller_id HAVING COUNT(*) >= 20
    )
""").df().iloc[0, 0]
print(f"Result grain: {len(q6)} rows -- bottom-20-per-category seller shortlist.")
print(f"{n_qualifying_sellers} sellers clear the >=20-order threshold overall.")
q6.head(15)

Result grain: 473 rows -- bottom-20-per-category seller shortlist.
804 sellers clear the >=20-order threshold overall.


,seller_id,primary_category,order_count,avg_review_score,pct_low_review,category_rank
0,e59aa562b9f8076dd550fcddf0e73491,agro_industry_and_commerce,69,4.261,13.04,1
1,d71d863e5ef30d94e440c11be17dcd8f,air_conditioning,23,3.333,47.62,1
2,6a51fc556dab5f766ced6fbc860bc613,air_conditioning,24,4.000,20.83,2
3,01cf7e3d21494c41fb86034f2e714fa1,air_conditioning,59,4.121,20.69,3
4,7a241947449cc45dbfda4f9d0798d9d0,air_conditioning,56,4.145,23.64,4
5,c31eff8334d6b3047ed34bebd4d62c36,art,120,3.874,30.25,1
6,7040e82f899a04d1b434b795a43b4617,audio,208,3.650,34.95,1
7,712e6ed8aa4aa1fa65dab41fed5737e4,auto,77,3.395,46.05,1
8,e21a1e8be4715b0fcfd5385643bc49c4,auto,20,3.700,40.00,2
9,cee48807215b30a12ca2ca10ffb5f250,auto,39,3.769,33.33,3


**Interpretation:** 804 sellers clear the ≥20-order threshold — comfortably enough for a category-ranked shortlist, not the "too few sellers hit the threshold" risk flagged as a possibility going in. The worst performers are severe, not marginal noise: `air_conditioning`'s bottom seller averages 3.33 with a 47.6% low-review rate on 23 orders, and `auto`'s worst seller posts 3.395/46.1% low-review on 77 orders — high enough volume that this is a real pattern, not a small-sample fluke. These bottom-ranked-per-category sellers are the concrete intervention shortlist Q6 is built to produce.

## Q7 — Freight-burden effect on review, category-controlled

**Business question:** Do customers punish Olist when freight feels disproportionate to product value — and is that effect real, or a category-mix artifact?

**Grain of result:** one row per `primary_category`.

**How to read it:** `freight_ratio = total_freight / total_price` per order. `NTILE(3) OVER (PARTITION BY primary_category ORDER BY freight_ratio)` buckets orders into terciles **relative to their own category** — so "high freight" means high for that category's peers, not high in absolute terms. `low_minus_high_gap = avg_review(low tercile) - avg_review(high tercile)`; larger gap = freight burden hurts more in that category.

**Cross-cutting rule:** explicitly satisfied — terciles are computed `PARTITION BY primary_category`, so no order is ever compared to a peer outside its own category.

In [12]:
Q7_COMMENT = """
-- Q7 -- Freight-burden effect on review, category-controlled
-- Business question: Do customers punish Olist when freight feels
--   disproportionate to product value -- and is that a real effect or a
--   category-mix artifact?
-- Grain of result: one row per primary_category.
-- How to read it: freight_ratio = total_freight / total_price per order.
--   NTILE(3) OVER (PARTITION BY primary_category ORDER BY freight_ratio) buckets
--   orders into terciles relative to their OWN category. low_minus_high_gap =
--   avg_review(low freight tercile) - avg_review(high freight tercile).
-- Cross-cutting note: terciles are computed PARTITION BY primary_category, so
--   no order is compared to a peer outside its own category.
"""

Q7_SQL = """
WITH freight AS (
    SELECT order_id, primary_category, review_score,
           total_freight / NULLIF(total_price, 0) AS freight_ratio
    FROM '{orders}'
    WHERE review_score IS NOT NULL AND total_price > 0
),
terciled AS (
    SELECT *, NTILE(3) OVER (PARTITION BY primary_category ORDER BY freight_ratio) AS freight_tercile
    FROM freight
),
category_tercile AS (
    SELECT primary_category, freight_tercile, COUNT(*) AS order_count, AVG(review_score) AS avg_review_score
    FROM terciled
    GROUP BY primary_category, freight_tercile
),
pivoted AS (
    SELECT
        primary_category,
        MAX(avg_review_score) FILTER (WHERE freight_tercile = 1) AS avg_review_low_freight,
        MAX(avg_review_score) FILTER (WHERE freight_tercile = 2) AS avg_review_mid_freight,
        MAX(avg_review_score) FILTER (WHERE freight_tercile = 3) AS avg_review_high_freight,
        SUM(order_count) AS total_orders
    FROM category_tercile
    GROUP BY primary_category
)
SELECT
    primary_category, total_orders,
    ROUND(avg_review_low_freight, 3) AS avg_review_low_freight,
    ROUND(avg_review_mid_freight, 3) AS avg_review_mid_freight,
    ROUND(avg_review_high_freight, 3) AS avg_review_high_freight,
    ROUND(avg_review_low_freight - avg_review_high_freight, 3) AS low_minus_high_gap
FROM pivoted
ORDER BY low_minus_high_gap DESC
""".format(orders=ORDERS_PATH)

register("Q7", Q7_COMMENT, Q7_SQL)

q7 = duckdb.sql(Q7_SQL).df()
freight_stats = duckdb.sql(f"""
    SELECT MIN(total_freight/total_price) AS min_ratio, MAX(total_freight/total_price) AS max_ratio,
           APPROX_QUANTILE(total_freight/total_price, 0.5) AS median_ratio,
           APPROX_QUANTILE(total_freight/total_price, 0.99) AS p99_ratio
    FROM '{ORDERS_PATH}' WHERE total_price > 0
""").df()
print(f"Result grain: {len(q7)} rows -- one row per category.")
print("freight_ratio range check:", freight_stats.to_dict("records")[0])
q7.head(15)

Result grain: 74 rows -- one row per category.
freight_ratio range check: {'min_ratio': 0.0, 'max_ratio': 21.447058823529414, 'median_ratio': 0.22494226337644377, 'p99_ratio': 1.4569160990970584}


,primary_category,total_orders,avg_review_low_freight,avg_review_mid_freight,avg_review_high_freight,low_minus_high_gap
0,pc_gamer,7.0,4.667,2.500,2.500,2.167
1,portateis_cozinha_e_preparadores_de_alimentos,13.0,4.400,3.500,2.750,1.650
2,fashion_sport,25.0,4.889,4.500,3.875,1.014
3,la_cuisine,12.0,4.750,4.000,4.000,0.750
4,tablets_printing_image,76.0,4.385,4.320,3.720,0.665
5,diapers_and_hygiene,25.0,4.111,4.125,3.625,0.486
6,food,432.0,4.639,4.188,4.181,0.458
7,audio,341.0,4.018,3.956,3.566,0.451
8,fashion_underwear_beach,116.0,4.103,4.231,3.684,0.418
9,home_confort,369.0,4.041,3.976,3.699,0.341


**Interpretation — outlier check + thin-data flag:** `freight_ratio` does have extreme outliers (max 21.4× the item price, vs. a median of 0.22 and a 99th percentile of 1.46), exactly the distortion risk called out going in — but bucketing by within-category **rank** (`NTILE`) rather than raw value makes the tercile split robust to those outliers; a few extreme ratios can't change which third of a category's orders count as "high freight" relative to peers. The gap is real but concentrated in low-volume categories: the top of the ranked list (`pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos`, `home_comfort_2`, `fashion_sport`, `la_cuisine`) all have well under 100 orders, so those headline gaps (up to 2.17 points) shouldn't be trusted. **Suggested adjustment:** restrict to categories with reasonable volume before ranking — doing that, `food` (432 orders, 0.46-point gap), `audio` (341 orders, 0.45-point gap), and `home_confort` (369 orders, 0.34-point gap) are the most credible cases of high relative freight hurting reviews — smaller than the headline numbers suggest, but a real, actionable signal in a handful of categories.

## Q8 — Product-size-adjusted delivery expectation gap

**Business question:** Which states and sellers are slow *relative to what's reasonable for the product type they ship*, not slow in absolute terms?

**Grain of result:** three outputs — (a) sanity-check of the weight-bucket cutoffs; (b) one row per state; (c) one row per seller (≥20 orders), top 20 slowest.

**How to read it:** products are bucketed Light/Medium/Heavy by overall percentiles of `product_weight_g` (bottom/middle/top third of all *order line items*, so a popular light product shipped 1,000 times counts 1,000 times — the shipment-weighted distribution, not the catalog distribution). `category_weight_median` uses `MEDIAN(actual_delivery_days) OVER (PARTITION BY category, weight_bucket)` — DuckDB's windowed `MEDIAN` is the README's allowed "equivalent windowed median" to `PERCENTILE_CONT(0.5) WITHIN GROUP`. `excess_delivery_days = actual_delivery_days - category_weight_median`; positive means slower than what's normal for that category+weight combination.

**Cross-cutting rule:** explicitly satisfied — the median baseline is computed `PARTITION BY category, weight_bucket`, so state/seller slowness is measured against a category-and-size-appropriate norm, not a single global delivery-time baseline.

In [13]:
Q8_COMMENT = """
-- Q8 -- Product-size-adjusted delivery expectation gap
-- Business question: Which states/sellers are slow relative to what's
--   reasonable for the product type they ship, not slow in absolute terms?
-- Grain of result: one row per order line item (base CTE chain below), then
--   aggregated to state and to seller (>= 20 orders) in the two queries after it.
-- How to read it: weight_bucket = Light/Medium/Heavy by overall percentiles of
--   product_weight_g (item-grain, i.e. shipment-weighted, not catalog-weighted).
--   category_weight_median = MEDIAN(actual_delivery_days) OVER (PARTITION BY
--   category, weight_bucket) -- DuckDB's windowed MEDIAN, used as the README's
--   allowed "equivalent windowed median" to PERCENTILE_CONT(0.5) WITHIN GROUP.
--   excess_delivery_days = actual_delivery_days - category_weight_median;
--   positive = slower than normal for that category+weight combination.
-- Cross-cutting note: the median baseline is PARTITION BY category,
--   weight_bucket, so slowness is measured against a category-and-size-
--   appropriate norm, not one global delivery-time baseline.
"""

Q8_BASE_CTE = """
WITH weight_buckets AS (
    SELECT *, NTILE(3) OVER (ORDER BY product_weight_g) AS weight_tercile
    FROM '{items}'
),
labeled AS (
    SELECT *, CASE weight_tercile WHEN 1 THEN 'Light' WHEN 2 THEN 'Medium' ELSE 'Heavy' END AS weight_bucket
    FROM weight_buckets
),
with_median AS (
    SELECT *,
        MEDIAN(actual_delivery_days) OVER (PARTITION BY category, weight_bucket) AS category_weight_median
    FROM labeled
),
with_excess AS (
    SELECT *, actual_delivery_days - category_weight_median AS excess_delivery_days
    FROM with_median
)
""".format(items=ITEMS_PATH)

Q8_CUTOFFS_SQL = Q8_BASE_CTE + """
SELECT weight_bucket, COUNT(*) AS n, MIN(product_weight_g) AS min_g, MAX(product_weight_g) AS max_g,
       ROUND(AVG(excess_delivery_days), 3) AS avg_excess_delivery_days
FROM with_excess
GROUP BY weight_bucket
ORDER BY min_g
"""

register("Q8_weight_cutoffs", Q8_COMMENT, Q8_CUTOFFS_SQL)

q8_cutoffs = duckdb.sql(Q8_CUTOFFS_SQL).df()
print(f"Result grain: {len(q8_cutoffs)} rows -- one row per weight bucket (sanity check on the tercile split).")
q8_cutoffs

Result grain: 3 rows -- one row per weight bucket (sanity check on the tercile split).


,weight_bucket,n,min_g,max_g,avg_excess_delivery_days
0,Light,36721,2.0,400.0,2.177
1,Medium,36721,400.0,1350.0,2.147
2,Heavy,36721,1350.0,40425.0,2.234


In [14]:
Q8_STATE_COMMENT = """
-- Q8 (continued) -- aggregated to state
-- Grain of result: one row per customer_state.
-- How to read it: avg_excess_delivery_days averaged across order LINE ITEMS in
--   that state (an order with 3 items from 3 sellers contributes 3 rows) --
--   consistent with this query's item-grain source per the Section 6 grain
--   reminder. slowness_rank ranks states from slowest to fastest.
"""

Q8_STATE_SQL = Q8_BASE_CTE + """
SELECT
    customer_state,
    COUNT(*) AS item_rows,
    COUNT(DISTINCT order_id) AS order_count,
    ROUND(AVG(excess_delivery_days), 2) AS avg_excess_delivery_days,
    RANK() OVER (ORDER BY AVG(excess_delivery_days) DESC) AS slowness_rank
FROM with_excess
GROUP BY customer_state
ORDER BY slowness_rank
"""

register("Q8_by_state", Q8_STATE_COMMENT, Q8_STATE_SQL)

q8_state = duckdb.sql(Q8_STATE_SQL).df()
print(f"Result grain: {len(q8_state)} rows -- one row per state.")
q8_state

Result grain: 27 rows -- one row per state.


,customer_state,item_rows,order_count,avg_excess_delivery_days,slowness_rank
0,AP,81,67,17.75,1
1,RR,46,41,17.67,2
2,AM,163,145,16.11,3
3,AL,427,397,13.89,4
4,PA,1054,946,13.15,5
5,MA,800,717,11.18,6
6,SE,375,335,10.99,7
7,CE,1426,1279,10.63,8
8,AC,91,80,10.19,9
9,PB,586,517,10.11,10


In [15]:
Q8_SELLER_COMMENT = """
-- Q8 (continued) -- aggregated to seller, minimum-volume filtered
-- Grain of result: one row per seller, sellers with >= 20 orders only (same
--   minimum-volume threshold as Q6, to avoid noise from tiny sellers).
-- How to read it: same excess_delivery_days definition as Q8_by_state.
--   Top 20 slowest sellers shown -- an intervention shortlist.
"""

Q8_SELLER_SQL = Q8_BASE_CTE + """
SELECT
    seller_id,
    COUNT(DISTINCT order_id) AS order_count,
    ROUND(AVG(excess_delivery_days), 2) AS avg_excess_delivery_days,
    RANK() OVER (ORDER BY AVG(excess_delivery_days) DESC) AS slowness_rank
FROM with_excess
GROUP BY seller_id
HAVING COUNT(DISTINCT order_id) >= 20
ORDER BY slowness_rank
LIMIT 20
"""

register("Q8_by_seller", Q8_SELLER_COMMENT, Q8_SELLER_SQL)

q8_seller = duckdb.sql(Q8_SELLER_SQL).df()
print(f"Result grain: {len(q8_seller)} rows -- top 20 slowest sellers with >=20 orders.")
q8_seller

Result grain: 20 rows -- top 20 slowest sellers with >=20 orders.


,seller_id,order_count,avg_excess_delivery_days,slowness_rank
0,66e0557ecc2b4dbea057e93f215f68d8,30,20.30,1
1,054694fa03fe82cec4b7551487331d74,20,19.52,2
2,2a1348e9addc1af5aaa619b1a3679d6b,48,16.98,3
3,54965bbe3e4f07ae045b90b0b8541f52,73,16.23,4
4,7f152321c60a266edc53af1925ef96c1,20,14.10,5
5,ad781527c93d00d89a11eecd9dcad7c1,38,14.03,6
6,712e6ed8aa4aa1fa65dab41fed5737e4,77,13.39,7
7,cee48807215b30a12ca2ca10ffb5f250,39,13.38,8
8,17f51e7198701186712e53a39c564617,56,12.21,9
9,8444e55c1f13cd5c179851e5ca5ebd00,92,11.27,10


**Interpretation:** weight terciles split at 400g and 1,350g (Light/Medium/Heavy), and the median-adjusted excess is small and similar across all three buckets (~2.1–2.2 days) — the category-and-weight-median control is working as intended; a right-skewed delivery-time distribution (mostly on-time, with a long tail of very late deliveries) pulls the *mean* excess above the *median* residual of 0, it isn't a sign the adjustment failed. At the state level the story is stark and directionally sensible: **SP**, where most sellers and carrier infrastructure are concentrated, delivers 1.5 days *faster* than the category-weight norm, while remote northern states **AP and RR** run 17–18 days slower. This isn't just small-state noise — mid-to-large states **BA** (3,255 orders, +8.75 days) and **PE** (1,593 orders, +7.88 days) confirm a genuine geography-driven slowness gap at scale. At the seller level, the 20 slowest among the 804 sellers clearing the ≥20-order bar run 9–20 days over their category-weight norm on 20–92 orders each — a concrete, product-mix-adjusted list for logistics follow-up.

## Export to `sql/queries.sql`

Writes every query above — with its comment block — to a standalone `.sql` file, in the same order they ran here, so it's readable and runnable independently of this notebook (e.g. `duckdb < sql/queries.sql` from the repo root, or pasted into a DuckDB CLI session).

In [16]:
SQL_OUTPUT_PATH = Path("sql/queries.sql")
SQL_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

header = """-- Olist Marketplace -- Phase 2 SQL Analysis
-- Generated from notebooks/02_sql_analysis.ipynb -- run it there for narrative
-- markdown, business-language interpretation, and result tables.
-- Standalone usage: duckdb < sql/queries.sql (from the repo root), or paste
-- individual statements into a DuckDB CLI session pointed at data/processed/.
-- Q5 was intentionally removed (folded into Phase 4's chi-square test).

"""

blocks = [header]
for label, comment_block, sql_text in exported_queries:
    blocks.append(comment_block + "\n" + sql_text + ";\n")

SQL_OUTPUT_PATH.write_text("\n".join(blocks), encoding="utf-8")
print(f"wrote {len(exported_queries)} queries to {SQL_OUTPUT_PATH}")
print([label for label, _, _ in exported_queries])

wrote 12 queries to sql\queries.sql
['Q0a_overall', 'Q0b_monthly', 'Q1', 'Q2', 'Q3', 'Q4_breakdown', 'Q4_overall_diff', 'Q6', 'Q7', 'Q8_weight_cutoffs', 'Q8_by_state', 'Q8_by_seller']


## SQL findings summary

- **21.1% of reviewed orders score ≤ 3** (95,824 reviewed orders out of 96,470 delivered), representing **R$2.78M of revenue-at-risk** out of R$13.22M total product revenue — roughly 1 in 5 orders, concentrated unevenly.
- **`bed_bath_table` is the #1 priority category** by revenue-at-risk (R$263k) purely on volume (9,181 orders) despite middling quality (25.7% low-review); `office_furniture` has the *worst* quality of any major category (36.3% low-review, avg score 3.65) but ranks only 10th on dollar exposure — revenue-at-risk and review-score-alone point to different "fix this first" answers.
- **Late delivery reliably hurts reviews everywhere**: even in the highest-volume categories, going from on-time to 8+ days late drops average review by ~2.4–2.6 points (`health_beauty` 2.63, `sports_leisure` 2.50, `auto` 2.42, `bed_bath_table` 2.38). The "early delivery helps" claim is *not* reliable at the category level — the exact on-time bucket is only 1.3% of orders, too thin to trust per category without redefining it as a wider band.
- **23 of 27 states show the expected OTD/review alignment**, but **PE and AC** have good on-time delivery *and* below-median reviews (delivery isn't their problem), while **MS and TO** tolerate below-median on-time rates without a review hit — four states where a pure logistics fix wouldn't be the right lever.
- **Seller tenure does not predict better reviews** — category-controlled, Established sellers (≥6 months observed) average **0.174 points lower** than New sellers, the opposite of an onboarding/experience story; this held within `bed_bath_table`, `computers_accessories`, and `auto` individually, and is flagged as a Phase 4 hypothesis to verify rather than a settled conclusion.
- **804 sellers clear the ≥20-order volume bar** for fair ranking; the worst Q6 sellers hit 40–48% low-review rates on 20–80+ orders each — a real, non-noisy intervention shortlist, not a "too few sellers" problem.
- **Freight-burden effect is real but concentrated in a handful of categories once thin (<100-order) categories are excluded**: `food` (432 orders, 0.46-pt low-vs-high-freight-tercile gap), `audio` (341 orders, 0.45 pt), and `home_confort` (369 orders, 0.34 pt) are the credible cases; the largest headline gaps (up to 2.17 pts) are in categories with well under 100 orders and shouldn't be trusted.
- **Delivery slowness compounds with geography even after controlling for product weight and category**: `SP` delivers 1.5 days faster than its category-weight norm, while `AP`/`RR` run 17–18 days slower — confirmed at scale by mid-size states `BA` (+8.75 days, 3,255 orders) and `PE` (+7.88 days, 1,593 orders), not just small-state noise.

These are the raw inputs for Phase 5's findings & recommendations section.